# Intervalos de incerteza — RUL FD001

Uma previsão pontual informa um valor, mas não mostra quanto essa estimativa pode variar. Nesta etapa, será construído um intervalo empírico em torno da previsão de RUL usando os erros de previsões fora da amostra. A calibração será balanceada por motor para evitar que trajetórias longas dominem o resultado.

A ideia é inspirada na predição conformal, que calibra intervalos a partir dos erros observados em dados de calibração (Angelopoulos; Bates, 2023). Como as observações do C-MAPSS são temporais e pertencem a motores diferentes, o resultado será tratado como uma referência empírica de incerteza, e não como uma garantia formal de cobertura.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from multilang import LangMap

pio.renderers.default = 'vscode'
lang = LangMap(source='pt', target='pt')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'dataset').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_eda import load_split
from cmapss_modeling import add_rul_label, candidate_rul_features, nonconstant_features
from cmapss_optuna import build_pipeline_from_params, optimize_model
from cmapss_early_warning import generate_oof_rul_predictions
from cmapss_evaluation import last_observation_per_unit, load_official_rul
from cmapss_uncertainty import (
    add_prediction_interval,
    calibrate_groupwise_interval,
    evaluate_interval_coverage,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'dataset'

## 1. Calibração dos erros por motor

Será reutilizado o Extra Trees v1. As previsões fora da amostra serão produzidas por grupos de motores, e uma largura de intervalo será obtida a partir dos erros absolutos observados nesses grupos. O nível nominal será de 90%, apenas como cenário de análise.

In [2]:
train = add_rul_label(load_split(DATA_DIR, 'train', 'FD001'))
test = load_split(DATA_DIR, 'test', 'FD001')
features = nonconstant_features(
    train,
    candidate_rul_features(include_settings=True),
)
study = optimize_model(
    train, features=features, model_name='Extra Trees',
    n_trials=100, n_splits=5, random_state=42,
    storage_dir=PROJECT_ROOT / 'outputs' / 'optuna',
    search_version='v1',
)
model = build_pipeline_from_params(
    'Extra Trees', study.best_params, len(features),
    random_state=42, search_version='v1',
)
oof_predictions = generate_oof_rul_predictions(
    train, features=features, model=model, n_splits=5,
)
half_width, motor_scores = calibrate_groupwise_interval(
    oof_predictions, coverage=0.90,
)
pd.DataFrame([{
    'cobertura_nominal': 0.90,
    'meia_largura_intervalo': half_width,
    'motores_calibracao': len(motor_scores),
}]).round(2)

[I 2026-09-26 17:09:41,552] Using an existing study with `study_name='fd001_extra_trees_v1'` instead of creating a new one.


,cobertura_nominal,meia_largura_intervalo,motores_calibracao
0,0.9,75.68,100


### Insights

A largura calculada resume a incerteza observada nos motores de treino. Ela não deve ser interpretada como uma margem física do motor; representa quanto o erro do modelo variou sob o protocolo de validação adotado. Uma largura grande indicará que a previsão pontual precisa ser acompanhada de cautela.

## 2. Intervalos no teste oficial

Depois da calibração, o modelo será treinado com todos os motores de treino e aplicado à última observação disponível de cada motor do conjunto oficial de teste. O RUL oficial será usado apenas para verificar a cobertura observada, sem alterar a largura calibrada.

In [3]:
model.fit(train[features], train['rul'])
last_test = last_observation_per_unit(test)
test_predictions = pd.DataFrame({
    'unit': last_test['unit'].to_numpy(),
    'cycle': last_test['cycle'].to_numpy(),
    'rul_real': load_official_rul(DATA_DIR, 'FD001').to_numpy(),
    'rul_predito': model.predict(last_test[features]),
})
test_with_interval = add_prediction_interval(test_predictions, half_width)
interval_metrics = evaluate_interval_coverage(
    test_with_interval,
    actual_column='rul_real',
)
pd.DataFrame([interval_metrics]).round(2)

,cobertura_observada,largura_media,observacoes
0,1.0,135.83,100.0


### Insights

A cobertura observada será comparada com o nível nominal de 90%, mas não será tratada como uma prova definitiva de qualidade estatística. O teste possui poucos motores independentes e as trajetórias são temporais; por isso, o intervalo deverá ser usado como sinal de cautela para a decisão, não como certeza matemática.

In [4]:
plot_data = test_with_interval.sort_values('unit')
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_data['unit'], y=plot_data['rul_superior'],
    mode='lines', line={'width': 0}, showlegend=False,
))
fig.add_trace(go.Scatter(
    x=plot_data['unit'], y=plot_data['rul_inferior'],
    mode='lines', fill='tonexty', fillcolor='rgba(99,110,250,0.22)',
    line={'width': 0}, name='Intervalo de referência',
))
fig.add_trace(go.Scatter(
    x=plot_data['unit'], y=plot_data['rul_real'],
    mode='markers', name='RUL real',
))
fig.add_trace(go.Scatter(
    x=plot_data['unit'], y=plot_data['rul_predito'],
    mode='markers', name='RUL previsto',
))
fig.update_layout(
    title=lang({'title': 'RUL previsto e intervalo de referência — teste FD001'})['title'],
    xaxis_title='Motor', yaxis_title='RUL (ciclos)', height=600,
)
fig.show()

## Referências

ANGELOPOULOS, A. N.; BATES, S. Conformal Prediction: A Gentle Introduction. Foundations and Trends in Machine Learning, v. 16, n. 4, p. 494–591, 2023.